# 01 — Dataset Audit

## Objective

Audit the structural integrity of the complete dataset and verify the correspondence between images and LabelMe annotations.

## Motivation

A reliable curation pipeline must start from readable files and unambiguous image-to-annotation relationships. Structural problems identified here would invalidate or distort every downstream analysis.

## Inputs

- Image archive configured in `curation/config.yaml`.
- Annotation archive configured in `curation/config.yaml`.

## Outputs

- `curation/outputs/01-dataset-audit/inventory.csv`
- `curation/outputs/01-dataset-audit/annotation_index.csv`
- `curation/outputs/01-dataset-audit/manifest.yaml`

Unmatched cases remain derivable from the two canonical tables and are not persisted separately.


In [ ]:
# Environment-specific setup
import pathlib
import sys

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../../')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

ROOT = base_folder.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from curation.common import load_config, prepare_dataset, stage_output_dir

CONFIG = load_config(ROOT)
DATASET = prepare_dataset(ROOT)
IMAGES_DIR = DATASET["images_dir"]
ANNOTATIONS_DIR = DATASET["annotations_dir"]

from curation.common import load_labelme, sha256_file, write_manifest

STAGE = "01-dataset-audit"
STAGE_DIR = stage_output_dir(STAGE, ROOT)


In [ ]:
from collections import defaultdict
import pandas as pd
from PIL import Image

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
images = sorted(path for path in IMAGES_DIR.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
annotation_files = sorted(path for path in ANNOTATIONS_DIR.rglob("*.json") if path.is_file())

annotation_rows = []
by_image_name = defaultdict(list)
by_stem = defaultdict(list)
for path in annotation_files:
    relative = path.relative_to(ANNOTATIONS_DIR).as_posix()
    try:
        value = load_labelme(path)
        image_path = str(value.get("imagePath", "") or "")
        image_name = Path(image_path.replace("\\", "/")).name
        labels = sorted({str(shape.get("label", "") or "").strip() for shape in value["shapes"]})
        parse_error = None
        is_labelme = True
    except Exception as error:
        image_path, image_name, labels = "", "", []
        parse_error = f"{type(error).__name__}: {error}"
        is_labelme = False
    row = {
        "annotation_relative_path": relative,
        "annotation_filename": path.name,
        "annotation_stem": path.stem,
        "is_labelme": is_labelme,
        "imagePath": image_path,
        "imagePath_filename": image_name,
        "n_shapes": len(value["shapes"]) if is_labelme else 0,
        "labels": "|".join(labels),
        "parse_error": parse_error,
        "matched_to_image": False,
    }
    annotation_rows.append(row)
    if is_labelme:
        if image_name:
            by_image_name[image_name.casefold()].append(relative)
        by_stem[path.stem.casefold()].append(relative)

annotation_index = pd.DataFrame(annotation_rows)

In [ ]:
matched = set()
inventory_rows = []
for image_id, path in enumerate(images, start=1):
    named = by_image_name.get(path.name.casefold(), [])
    stemmed = by_stem.get(path.stem.casefold(), [])
    if len(named) == 1:
        candidates, method = named, "labelme_imagePath"
    elif len(stemmed) == 1:
        candidates, method = stemmed, "filename_stem"
    else:
        candidates, method = sorted(set(named + stemmed)), None
    annotation_relative = candidates[0] if len(candidates) == 1 else None
    if annotation_relative:
        matched.add(annotation_relative)

    readable, width, height, mode, image_error = False, None, None, None, None
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            width, height, mode = image.width, image.height, image.mode
        readable = True
    except Exception as error:
        image_error = f"{type(error).__name__}: {error}"

    inventory_rows.append({
        "image_id": image_id,
        "relative_path": path.relative_to(IMAGES_DIR).as_posix(),
        "filename": path.name,
        "stem": path.stem,
        "extension": path.suffix.lower(),
        "file_size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "readable": readable,
        "width": width,
        "height": height,
        "mode": mode,
        "annotation_relative_path": annotation_relative,
        "annotation_match_method": method,
        "n_annotation_candidates": len(candidates),
        "image_error": image_error,
    })

inventory = pd.DataFrame(inventory_rows)
annotation_index["matched_to_image"] = annotation_index["annotation_relative_path"].isin(matched)
inventory_path = STAGE_DIR / "inventory.csv"
annotation_index_path = STAGE_DIR / "annotation_index.csv"
inventory.to_csv(inventory_path, index=False)
annotation_index.to_csv(annotation_index_path, index=False)
display(inventory.head())

In [ ]:
summary = {
    "images": int(len(inventory)),
    "annotations": int(len(annotation_index)),
    "labelme_annotations": int(annotation_index["is_labelme"].sum()),
    "unreadable_images": int((~inventory["readable"]).sum()),
    "images_without_unique_annotation": int(inventory["annotation_relative_path"].isna().sum()),
    "annotations_without_image": int((~annotation_index["matched_to_image"]).sum()),
    "labels_found": sorted({label for labels in annotation_index["labels"].fillna("") for label in labels.split("|") if label}),
}
write_manifest(
    STAGE,
    "01-dataset-audit.ipynb",
    inputs={"image_archive": DATASET["image_archive"], "annotation_archive": DATASET["annotation_archive"]},
    parameters={"image_extensions": sorted(IMAGE_EXTENSIONS), "annotation_match_order": ["labelme_imagePath", "filename_stem"]},
    artifacts=[inventory_path, annotation_index_path],
    summary=summary,
    repo_root=ROOT,
)
summary